<a href="https://colab.research.google.com/github/BubuDavid/ml-journey-workshop/blob/main/notebooks/gpt2_attention_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 GPT-2: Attention en Acción

En este notebook vamos a explorar la arquitectura de GPT-2 de forma práctica. La idea central es simple:

1. **Tomar un modelo GPT-2** con pesos aleatorios (sin entrenamiento) y ver qué produce.
2. **Cargar los pesos entrenados** y ver la diferencia.
3. En ambos casos, **visualizar las matrices de atención** para entender qué está "mirando" el modelo.
4. Comparar **greedy decoding vs. sampling con temperatura**.

### Prerequisitos
- Álgebra lineal básica: multiplicación de matrices, softmax.
- Haber escuchado los términos: *attention*, *attention head*, *multi-head attention*, *transformer*.

### Arquitectura rápida de GPT-2 (small)
- **12 capas** (transformer blocks)
- **12 attention heads** por capa
- **768 dimensiones** de embedding
- Cada head opera sobre un subespacio de 768/12 = **64 dimensiones**

Cada attention head produce una **matriz de atención** de forma `(seq_len, seq_len)` que nos dice, para cada token, cuánto "atiende" a los demás tokens previos.

---
## 1. Setup

In [ ]:
!pip install transformers torch -q

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import copy
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports listos")

In [ ]:
# Cargar tokenizer y modelo preentrenado
tokenizer = GPT2Tokenizer.from_pretrained("gpt2", output_attentions=True)
model_pretrained = GPT2LMHeadModel.from_pretrained("gpt2", output_attentions=True)
model_pretrained.eval()

print(f"Vocabulario: {tokenizer.vocab_size:,} tokens")
print(f"Parámetros del modelo: {sum(p.numel() for p in model_pretrained.parameters()):,}")
print(f"Capas: {model_pretrained.config.n_layer}")
print(f"Heads por capa: {model_pretrained.config.n_head}")
print(f"Dimensión de embedding: {model_pretrained.config.n_embd}")

---
## 2. Modelo con pesos aleatorios: generación de texto

Vamos a crear una copia del modelo y **aleatorizar todos sus pesos**. Esto es como tener la arquitectura correcta (las "tuberías" del transformer) pero sin ningún conocimiento aprendido del lenguaje. ¿Qué texto producirá?

In [ ]:
# Crear modelo con pesos aleatorios usando la misma configuración
config = GPT2Config.from_pretrained("gpt2", output_attentions=True)
model_random = GPT2LMHeadModel(config)  # Inicialización aleatoria por default
model_random.eval()

print("✅ Modelo con pesos aleatorios creado")
print(f"   Misma arquitectura: {model_random.config.n_layer} capas, {model_random.config.n_head} heads")

In [ ]:
def generate_text_greedy(model, tokenizer, prompt, max_new_tokens=40):
    """
    Generación greedy: en cada paso, elegimos el token con la
    probabilidad más alta. Determinístico.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids)
            # outputs.logits shape: (batch, seq_len, vocab_size)
            next_token_logits = outputs.logits[:, -1, :]  # logits del último token
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token_id], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
# --- CONFIGURABLE ---
GENERATION_PROMPT = "The meaning of life is"
# --------------------

print("=" * 60)
print("PESOS ALEATORIOS — Greedy Decoding")
print("=" * 60)
print(f"Prompt: '{GENERATION_PROMPT}'")
print()
text_random = generate_text_greedy(model_random, tokenizer, GENERATION_PROMPT)
print(text_random)

☝️ **Observa el output.** El modelo produce texto sin sentido — caracteres repetidos, palabras incoherentes, estructura nula. La arquitectura está ahí, pero los pesos no codifican ningún patrón del lenguaje.

Esto tiene sentido: la atención está operando sobre representaciones que no significan nada.

---
## 3. Attention con pesos aleatorios

Ahora veamos **qué están haciendo las matrices de atención** con pesos aleatorios. Recordemos que cada head produce una matriz `(seq_len, seq_len)` donde la entrada `[i, j]` nos dice cuánta atención le pone el token `i` al token `j`.

Con pesos aleatorios, esperamos ver patrones **difusos y sin estructura** — el modelo no sabe a qué prestar atención.

In [ ]:
def get_attention_maps(model, tokenizer, text):
    """
    Pasa un texto por el modelo y regresa todas las matrices de atención.

    Returns:
        attentions: tuple de tensores, uno por capa.
                    Cada tensor tiene shape (1, n_heads, seq_len, seq_len)
        tokens: lista de strings, los tokens del input.
    """
    input_ids = tokenizer.encode(text, return_tensors="pt")
    tokens = [tokenizer.decode(t) for t in input_ids[0]]

    with torch.no_grad():
        outputs = model(input_ids, output_attentions=True)

    return outputs.attentions, tokens

In [ ]:
def plot_attention_heads(attentions, tokens, layer_idx, title_prefix=""):
    """
    Visualiza los 12 attention heads de una capa como heatmaps.

    La matriz de atención es triangular inferior porque GPT-2 usa
    causal masking: cada token solo puede atender a los tokens
    anteriores (y a sí mismo), nunca a los futuros.

    Args:
        attentions: tuple de tensores de atención (output del modelo)
        tokens: lista de tokens como strings
        layer_idx: índice de la capa a visualizar (0-11)
        title_prefix: prefijo para el título del plot
    """
    n_heads = attentions[layer_idx].shape[1]
    attn = attentions[layer_idx][0]  # (n_heads, seq_len, seq_len)

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    fig.suptitle(
        f"{title_prefix}Capa {layer_idx} — 12 Attention Heads",
        fontsize=16, fontweight="bold"
    )

    for head_idx in range(n_heads):
        ax = axes[head_idx // 4, head_idx % 4]
        head_attn = attn[head_idx].numpy()

        im = ax.imshow(head_attn, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"Head {head_idx}", fontsize=10)
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=7)
        ax.set_yticklabels(tokens, fontsize=7)

    fig.colorbar(im, ax=axes, shrink=0.6, label="Attention weight")
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

In [ ]:
# --- CONFIGURABLE ---
ATTENTION_PROMPT = "The cat sat on the couch"
LAYER_IDX = 0  # Cambia esto para explorar otras capas (0-11)
# --------------------

print(f"Prompt para atención: '{ATTENTION_PROMPT}'")
print(f"Visualizando capa: {LAYER_IDX}")
print()

attentions_random, tokens = get_attention_maps(model_random, tokenizer, ATTENTION_PROMPT)
plot_attention_heads(attentions_random, tokens, LAYER_IDX, title_prefix="🎲 Pesos Aleatorios — ")

☝️ **¿Qué ves?** Con pesos aleatorios, la atención se distribuye de forma más o menos uniforme — ningún head ha aprendido a especializarse. No hay patrones claros como "atender al token anterior" o "atender al inicio de la oración".

Cada fila de la matriz suma 1 (es un softmax), pero la distribución es casi plana.

---
## 4. Modelo con pesos entrenados: generación de texto

Ahora usemos el modelo **preentrenado** — el mismo GPT-2 pero con los pesos que aprendió de millones de páginas de texto. Misma arquitectura, misma cantidad de parámetros, pero ahora codifican patrones reales del lenguaje.

In [ ]:
print("=" * 60)
print("PESOS ENTRENADOS — Greedy Decoding")
print("=" * 60)
print(f"Prompt: '{GENERATION_PROMPT}'")
print()
text_pretrained = generate_text_greedy(model_pretrained, tokenizer, GENERATION_PROMPT)
print(text_pretrained)

print()
print("=" * 60)
print("COMPARACIÓN DIRECTA")
print("=" * 60)
print(f"Aleatorio:  {text_random}")
print(f"Entrenado:  {text_pretrained}")

☝️ **La diferencia es dramática.** El modelo entrenado produce texto coherente y gramaticalmente correcto. Los mismos ~124 millones de parámetros, la misma operación de multiplicación de matrices en cada attention head — pero los valores numéricos en esas matrices hacen toda la diferencia.

---
## 5. Attention con pesos entrenados

Ahora el momento clave: visualicemos la atención del modelo entrenado con **el mismo prompt y la misma capa** que usamos con los pesos aleatorios.

In [ ]:
attentions_pretrained, tokens = get_attention_maps(model_pretrained, tokenizer, ATTENTION_PROMPT)
plot_attention_heads(attentions_pretrained, tokens, 1, title_prefix="✅ Pesos Entrenados — ")

☝️ **¿Ves la diferencia?** Ahora cada head ha aprendido un **patrón específico**. Algunos patrones comunes que puedes observar:

- **Atención al token anterior**: una diagonal fuerte justo debajo de la diagonal principal.
- **Atención al primer token**: una columna brillante en la posición 0 (funciona como un "token basura" donde el modelo dummpea atención residual).
- **Atención posicional**: patrones que dependen de la distancia entre tokens.

Cada head se ha especializado durante el entrenamiento para capturar un tipo diferente de relación entre tokens. Eso es la gracia del **multi-head attention**: en lugar de una sola función de atención, tienes 12 "perspectivas" diferentes operando en paralelo.

In [ ]:
# Comparación lado a lado: un solo head
HEAD_IDX = 0  # Cambia esto para comparar otros heads

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, attn, label in zip(
    axes,
    [attentions_random, attentions_pretrained],
    ["🎲 Aleatorio", "✅ Entrenado"]
):
    head_attn = attn[LAYER_IDX][0, HEAD_IDX].numpy()
    im = ax.imshow(head_attn, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"{label}\nCapa {LAYER_IDX}, Head {HEAD_IDX}", fontsize=12)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.8, label="Attention weight")
fig.suptitle("Comparación: Pesos Aleatorios vs. Entrenados", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---
## 6. Explorando otras capas

Las capas tempranas (0-3) tienden a capturar patrones locales y sintácticos. Las capas más profundas (8-11) capturan relaciones más abstractas y semánticas. Cambia `LAYER_IDX` para verificarlo.

In [ ]:
# Comparación rápida: capa 0 vs capa 11 (pesos entrenados)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, layer, label in zip(axes, [0, 11], ["Capa 0 (shallow)", "Capa 11 (deep)"]):
    # Promedio de todos los heads para tener una vista resumen
    avg_attn = attentions_pretrained[layer][0].mean(dim=0).numpy()
    im = ax.imshow(avg_attn, cmap="Blues", vmin=0)
    ax.set_title(f"{label}\n(promedio de 12 heads)", fontsize=12)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.8, label="Avg attention weight")
fig.suptitle("✅ Pesos Entrenados — Capa Superficial vs. Profunda", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

---
## 7. Bonus: Greedy vs. Sampling con Temperatura

Hasta ahora usamos **greedy decoding**: siempre elegimos el token con mayor probabilidad. Esto es determinístico pero puede producir texto repetitivo.

Una alternativa es **sampling con temperatura**. La idea es simple: antes de aplicar softmax a los logits para obtener probabilidades, los dividimos entre un valor $T$ (la temperatura):

$$P(x_i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

- **T → 0**: el softmax se vuelve más "picky" — casi todo el peso va al token más probable (se acerca a greedy).
- **T = 1**: distribución original, sin modificar.
- **T > 1**: distribución más uniforme — el modelo se vuelve más "creativo" (y más impredecible).

In [ ]:
def generate_text_temperature(model, tokenizer, prompt, max_new_tokens=40, temperature=1.0):
    """
    Generación con sampling basado en temperatura.

    En lugar de siempre elegir el token más probable (greedy),
    muestreamos de la distribución de probabilidad escalada por T.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids)
            next_token_logits = outputs.logits[:, -1, :]

            # Escalar logits por temperatura
            scaled_logits = next_token_logits / temperature

            # Convertir a probabilidades y muestrear
            probs = F.softmax(scaled_logits, dim=-1)
            next_token_id = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token_id], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.5]

print("=" * 60)
print("PESOS ENTRENADOS — Efecto de la Temperatura")
print("=" * 60)
print(f"Prompt: '{GENERATION_PROMPT}'")
print()

print("[Greedy (T→0)]")
print(text_pretrained)
print()

for temp in temperatures:
    text = generate_text_temperature(model_pretrained, tokenizer, GENERATION_PROMPT, temperature=temp)
    print(f"[T = {temp}]")
    print(text)
    print()

☝️ **Observa cómo cambia el texto:**

- **T = 0.3**: Muy similar a greedy — el modelo es conservador y predecible.
- **T = 0.7**: Un buen balance — coherente pero con variación.
- **T = 1.0**: La distribución original — más diverso.
- **T = 1.5**: El modelo se vuelve "demasiado creativo" — empieza a perder coherencia.

La temperatura no cambia los pesos del modelo ni la atención — solo modifica **cómo seleccionamos el siguiente token** a partir de la distribución de probabilidad.

In [ ]:
# Visualización: distribución de probabilidad del siguiente token para diferentes temperaturas
input_ids = tokenizer.encode(GENERATION_PROMPT, return_tensors="pt")

with torch.no_grad():
    outputs = model_pretrained(input_ids)
    logits = outputs.logits[0, -1, :]  # logits del último token

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
top_k = 15  # Mostrar solo los top-k tokens

for ax, temp in zip(axes, temperatures):
    probs = F.softmax(logits / temp, dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k)
    top_tokens = [tokenizer.decode(idx) for idx in top_indices]

    bars = ax.barh(range(top_k), top_probs.numpy(), color="steelblue")
    ax.set_yticks(range(top_k))
    ax.set_yticklabels(top_tokens, fontsize=8)
    ax.set_xlabel("Probabilidad")
    ax.set_title(f"T = {temp}", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

fig.suptitle(
    f"Distribución del siguiente token después de: '{GENERATION_PROMPT}'",
    fontsize=13, fontweight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

☝️ **Esto es la temperatura en acción visual:**

- Con **T baja**, casi toda la probabilidad se concentra en un solo token.
- Con **T alta**, la probabilidad se reparte más uniformemente — el modelo tiene más opciones "razonables" de las cuales muestrear.

Nota que esto es exactamente la misma operación de **softmax** que se usa dentro de la atención — ahí el softmax decide cuánto peso darle a cada token previo; aquí decide cuánto peso darle a cada posible siguiente token.

---
## Resumen

| | Pesos Aleatorios | Pesos Entrenados |
|---|---|---|
| **Texto generado** | Gibberish incoherente | Texto fluido y coherente |
| **Matrices de atención** | Difusas, sin estructura | Patrones especializados por head |
| **Cada head aprende** | Nada | Un tipo específico de relación |

**Puntos clave:**
- La **arquitectura** (transformer, multi-head attention) define la estructura de cómputo.
- Los **pesos** (aprendidos durante el entrenamiento) determinan qué computa esa estructura.
- **Multi-head attention** permite que el modelo capture múltiples tipos de relaciones en paralelo.
- La **temperatura** controla la diversidad del output sin modificar el modelo.

In [ ]:
# 🔬 Playground: cambia estos valores y re-ejecuta las celdas relevantes
# GENERATION_PROMPT = "Once upon a time"
# ATTENTION_PROMPT = "She walked to the"
# LAYER_IDX = 6
# HEAD_IDX = 3